For Website:
The chart shows how U.S. foreign aid is distributed across different sectors in three regions every five years. The security and other sectors receive the most funding, especially in South & Central Asia, where these two categories dominate the aid distribution—particularly in 2010 and 2015. This indicates a strong focus on military, defense, and broader stability initiatives in that region during those years.
In Europe & Eurasia, aid is more evenly spread but still leans toward security and other sectors. There are also smaller but consistent investments in health, education, and democracy and governance, suggesting ongoing support for social services and institutional development.
In East Asia & Oceania, overall aid levels are significantly lower, though the money is still mainly directed toward the other, security, and health sectors. This suggests that while this region receives less funding, the focus is still on maintaining stability and meeting essential needs.
Overall, the data suggests that U.S. foreign aid priorities are strongly influenced by geopolitical strategy, with large portions going to security-related efforts and flexible or miscellaneous support.

Design Choices:
The design uses faceted bar charts to clearly compare how aid is distributed across sectors in different regions over time. Grouping the data by region allows for side-by-side comparison, while using stacked bars highlights the proportion of aid allocated to each sector within a given year. Selecting every five years simplifies the timeline and avoids clutter, making long-term trends easier to spot. Consistent color coding for sectors helps viewers quickly identify patterns across regions, and the vertical layout emphasizes the scale of aid differences. Overall, the design balances detail with clarity, making the complex data more digestible.



In [ ]:
!pip install altair vega_datasets ipywidgets


In [ ]:
import pandas as pd
import altair as alt

# Load and clean data
df = pd.read_csv('us_foreign_aid_funding.csv')
df = df[['Country Name', 'Funding Account Name', 'Transaction Type Name', 'Fiscal Year', 'current_amount']]
df = df[df['current_amount'] > 0]

# Assign regions
def assign_region(country):
    europe_eurasia = ['Albania', 'Armenia', 'Azerbaijan', 'Belarus', 'Bosnia and Herzegovina',
                      'Bulgaria', 'Croatia', 'Czechia', 'Estonia', 'Georgia', 'Hungary',
                      'Kazakhstan', 'Kosovo', 'Moldova', 'Montenegro', 'North Macedonia',
                      'Russia', 'Serbia', 'Ukraine']
    east_asia_oceania = ['China', 'Indonesia', 'Cambodia', 'Timor-Leste']
    south_central_asia = ['Afghanistan', 'Bangladesh', 'India', 'Pakistan']
    if country in europe_eurasia:
        return 'Europe & Eurasia'
    elif country in east_asia_oceania:
        return 'East Asia & Oceania'
    elif country in south_central_asia:
        return 'South & Central Asia'
    return None

df['Region'] = df['Country Name'].apply(assign_region)
df = df[df['Region'].notna()]

# Assign sectors
def assign_sector(name):
    name = str(name).lower()
    if 'health' in name or 'medical' in name or 'hiv' in name:
        return 'Health'
    if 'education' in name:
        return 'Education'
    if 'capital' in name or 'infrastructure' in name:
        return 'Infrastructure'
    if 'security' in name or 'defense' in name:
        return 'Security'
    if 'democracy' in name or 'governance' in name:
        return 'Democracy & Governance'
    return 'Other'

df['Sector'] = df['Funding Account Name'].apply(assign_sector)

# Aggregate
agg_df = df.groupby(['Fiscal Year', 'Region', 'Sector'])['current_amount'].sum().reset_index()

# Filter to every 5 years (e.g., 2000, 2005, ...)
years = sorted(agg_df['Fiscal Year'].unique())
filtered_years = [y for y in years if y % 5 == 0]
agg_df = agg_df[agg_df['Fiscal Year'].isin(filtered_years)]

# Plot
chart = alt.Chart(agg_df).mark_bar().encode(
    x=alt.X('Fiscal Year:O', title='Fiscal Year'),
    y=alt.Y('current_amount:Q', title='Aid Amount (USD)'),
    color=alt.Color('Sector:N', title='Sector'),
    column=alt.Column('Region:N', title='Region'),
    tooltip=['Region', 'Sector', 'Fiscal Year', 'current_amount']
).properties(
    width=150,
    height=400,
    title='U.S. Foreign Aid by Sector and Region (Every 5 Years)'
)

chart.configure_axis(labelAngle=0)
